# 20260529_circle_flow_acceleration

This notebook loads the simulated circle-flow acceleration experiment output from `output/20260529_circle_flow_acceleration/circle_flow_acceleration_experiments.csv`, computes end-of-trial per-robot summaries, and reproduces the same core plots as the reaction-delay notebook while comparing both acceleration limits and reaction delays.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# sns.set_theme(style="whitegrid")

In [ ]:
EXPERIMENT_NAME = "20260529_circle_flow_acceleration"
EXPERIMENT_DURATION_MIN = 5.0
ACCELERATION_ORDER = ["unlimited", "0.05"]
REACTION_DELAY_ORDER = ["0.00", "0.05"]

project_root = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
csv_path = project_root / "output" / EXPERIMENT_NAME / "circle_flow_acceleration_experiments.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Could not find experiment CSV at {csv_path}")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Match the physical experiment naming more closely
df["_time"] = df["time"]
df["laps_float"] = df["lap_count_ccw"]
df["target_speed_rad"] = df["angular_speed"]
df["max_speedup_acceleration"] = pd.to_numeric(df["max_speedup_acceleration"], errors="coerce")
df["reaction_delay"] = pd.to_numeric(df["reaction_delay"], errors="coerce")
df["accel_label"] = df["max_speedup_acceleration"].map(lambda x: f"{x:.2f}" if pd.notnull(x) else "unlimited")
df["reaction_delay_label"] = df["reaction_delay"].map(lambda x: f"{x:.2f}")
df["condition_label"] = (
    "accel=" + df["accel_label"] + ", delay=" + df["reaction_delay_label"]
)

df_last = (
    df
    .sort_values(["reaction_delay_label", "accel_label", "num_robots", "trial", "robot_id", "_time"])
    .groupby(["reaction_delay_label", "accel_label", "num_robots", "trial", "robot_id"], as_index=False)
    .tail(1)
)

df_last["laps_per_min"] = df_last["laps_float"] / EXPERIMENT_DURATION_MIN
df_last["target_laps_per_min"] = df_last["target_speed_rad"] / (2 * np.pi) * 60.0

min_rad_per_s = 0.3
max_rad_per_s = 0.6
min_laps_per_min = min_rad_per_s * 60.0 / (2 * np.pi)
max_laps_per_min = max_rad_per_s * 60.0 / (2 * np.pi)

theoretical_mean = (max_laps_per_min + min_laps_per_min) / 2.0
theoretical_variance = (max_laps_per_min - min_laps_per_min) ** 2 / 12.0

print(f"Loaded {len(df):,} logged rows")
print(f"Last-snapshot rows: {len(df_last):,}")
print("Acceleration limits present:", sorted(df_last["accel_label"].dropna().unique(), key=lambda x: ACCELERATION_ORDER.index(x)))
print("Reaction delays present:", sorted(df_last["reaction_delay_label"].dropna().unique(), key=lambda x: REACTION_DELAY_ORDER.index(x)))
df_last.head()

In [ ]:
df_last["accel_label"].dropna().unique()

In [ ]:
fig, axes = plt.subplots(len(REACTION_DELAY_ORDER), len(ACCELERATION_ORDER), figsize=(14, 10), sharex=True, sharey=True)
axes = np.atleast_2d(axes)

xs = np.linspace(min_laps_per_min, max_laps_per_min, 50)

for row_idx, reaction_delay_label in enumerate(REACTION_DELAY_ORDER):
    for col_idx, accel_label in enumerate(ACCELERATION_ORDER):
        ax = axes[row_idx, col_idx]
        plot_df = df_last[(df_last["reaction_delay_label"] == reaction_delay_label) & (df_last["accel_label"] == accel_label)]
        sns.scatterplot(
            data=plot_df,
            x="target_laps_per_min",
            y="laps_per_min",
            hue="num_robots",
            palette="tab20",
            alpha=0.9,
            ax=ax,
        )
        ax.plot(xs, xs, label="y = x", linestyle="--", color="gray")
        ax.set_title(f"accel = {accel_label}, delay = {reaction_delay_label} s")
        ax.set_xlabel("Target Laps Per Min")
        ax.set_ylabel("Actual Laps Per Min\n(For one robot over a full trial)")
        ax.set_ylim([0, None])
        ax.legend(title="num_robots", ncol=2, fontsize=8)

fig.suptitle("Simulation Data: Target vs Actual Laps Per Min by Acceleration Limit and Reaction Delay", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
df_sub = df_last.dropna(subset=["accel_label", "reaction_delay_label", "num_robots", "laps_per_min"])

stats = (
    df_sub
    .groupby(["reaction_delay_label", "accel_label", "num_robots"])
    .agg(
        mean=("laps_per_min", "mean"),
        var=("laps_per_min", "var"),
        std=("laps_per_min", "std"),
        count=("laps_per_min", "count"),
        distinct_runs=("trial", "nunique"),
    )
    .reset_index()
    .sort_values(["reaction_delay_label", "accel_label", "num_robots"])
)

stats["cv"] = stats["std"] / stats["mean"]
stats["condition_label"] = (
    "accel=" + stats["accel_label"] + ", delay=" + stats["reaction_delay_label"]
)
stats

In [ ]:
condition_order = [
    f"accel={accel}, delay={delay}"
    for delay in REACTION_DELAY_ORDER
    for accel in ACCELERATION_ORDER
]
palette = dict(zip(condition_order, sns.color_palette("tab10", n_colors=len(condition_order))))

fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharex=True)
fig.suptitle("Simulation Data: Laps Per Minute Summary Statistics by Acceleration Limit and Reaction Delay", fontsize=14)

for condition_label in condition_order:
    condition_stats = stats[stats["condition_label"] == condition_label].sort_values("num_robots")
    color = palette[condition_label]

    ax[0].errorbar(
        condition_stats["num_robots"],
        condition_stats["mean"],
        yerr=condition_stats["mean"] / np.sqrt(condition_stats["count"]),
        fmt="o-",
        capsize=4,
        color=color,
        label=condition_label,
    )
    condition_points = df_last[df_last["condition_label"] == condition_label]
    ax[0].scatter(condition_points["num_robots"], condition_points["laps_per_min"], alpha=0.08, color=color)

    sns.lineplot(
        data=condition_stats,
        x="num_robots",
        y="var",
        marker="o",
        ax=ax[1],
        color=color,
        label=condition_label,
    )

    sns.lineplot(
        data=condition_stats,
        x="num_robots",
        y="cv",
        marker="o",
        ax=ax[2],
        color=color,
        label=condition_label,
    )

ax[0].axhline(theoretical_mean, label="theoretical freeflow mean", linestyle="--", color="gray")
ax[0].set_title("Simulation: Mean laps_per_min vs num_robots")
ax[0].set_xlabel("num_robots")
ax[0].set_ylabel("mean(laps_per_min)")
ax[0].set_ylim([0, None])
ax[0].legend(fontsize=8)

ax[1].axhline(theoretical_variance, label="theoretical freeflow variance", linestyle="--", color="gray")
ax[1].set_title("Simulation: Variance of laps_per_min vs num_robots")
ax[1].set_xlabel("num_robots")
ax[1].set_ylabel("Var(laps_per_min)")
ax[1].set_ylim([0, None])
ax[1].legend(fontsize=8)

ax[2].set_title("Simulation: Coefficient of variation vs num_robots")
ax[2].set_xlabel("num_robots")
ax[2].set_ylabel("CV = std / mean")
ax[2].set_ylim([0, None])
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
stats[["reaction_delay_label", "accel_label", "num_robots", "mean", "var", "std", "count", "distinct_runs", "cv"]]

## Import experiment data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
disconnection_uuids = [
    "5b9197f4-c571-43fb-aa7d-594dfe98deba",
    "7ed870e1-8007-4313-a8ba-418d684e4583",
    "a4e4ecaf-f934-415e-9449-f87e8018c0fa",
    "fe06c0cb-c9e5-464b-833c-4b195ad0ebb4",
]

# maybe later need to id/verify these using a reproducible, coded way instead of via human judgment
jam_uuids = [
    "f171674d-8e08-4425-a580-94dadd118f5e",
    "e7b87a33-c0a4-4f2a-8d6c-2089764b12bc",
    "fd2fbb8e-21ca-43e3-a612-b412ccfecd20",
]

In [ ]:
# df = pd.read_csv("data_from_influx/influx_plus_recovery_notfiltered_20260511_1s.csv")

df = pd.read_csv("/Users/lucyliu/Documents/phd/traffic/robots/2026_passing_experiments/analysis/data_from_influx/influx_plus_recovery_notfiltered_20260511_1s.csv")

df = df[~df["run_uuid"].isin(disconnection_uuids + jam_uuids)]
df['_time'] = pd.to_datetime(df['_time'])
df['num_robots'] = df['num_robots'].astype(int)
df['r_current'] = np.sqrt(df['x']**2 + df['y']**2)

In [ ]:
df_last = (
    df
    .sort_values('_time')  # sort chronologically
    .groupby(['run_uuid', 'robot_id'], as_index=False)
    .tail(1)               # take the last row in each group
)

df_last['laps_per_min'] = df_last['laps_float'] / 5.0
df_last['target_laps_per_min'] = df_last['target_speed_rad'] / (2 * np.pi) * 60.0

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.scatterplot(
    # data=df_last[df_last["num_robots"] == 1],
    data = df_last,
    x="target_laps_per_min",
    y="laps_per_min",
    hue="num_robots",        # distinct colors per num_robots
    palette="tab20",       # or any palette you like
    alpha = 0.9
)

xs = np.linspace(min_laps_per_min, max_laps_per_min, 50)
plt.plot(xs, xs, label = "y = x", linestyle = "--", color = "gray")

plt.xlabel("Target Laps Per Min")
plt.ylabel("Actual Laps Per Min (For one robot over a full trial)")
plt.legend(title="num_robots", )
# plt.legend('off')
# plt.xlim([0, None])
plt.legend(ncol=3) 
plt.ylim([0, None])
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# # Filter to valid rows
# df_sub = df_last.dropna(subset=['num_robots', 'laps_per_min'])

# # Compute mean, variance, std, count, and coefficient of variation for each num_robots
# stats = (
#     df_sub
#     .groupby('num_robots')['laps_per_min']
#     .agg(mean='mean', var='var', std='std', count='count')
#     .reset_index()
#     .sort_values('num_robots')
# )

# Filter to valid rows
df_sub = df_last.dropna(subset=['num_robots', 'laps_per_min'])

# Compute stats including number of distinct run_UUIDs
stats = (
    df_sub
    .groupby('num_robots')
    .agg(
        mean=('laps_per_min', 'mean'),
        var=('laps_per_min', 'var'),
        std=('laps_per_min', 'std'),
        count=('laps_per_min', 'count'),
        distinct_runs=('run_uuid', 'nunique'),  # <- new column
    )
    .reset_index()
    .sort_values('num_robots')
)

# Coefficient of variation = std / mean
stats['cv'] = stats['std'] / stats['mean']

# 3-panel plot: mean (with error bars), variance, and CV vs num_robots
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

# 1) Mean with error bars (use std or std/sqrt(n) depending on what you want)
ax[0].errorbar(
    stats['num_robots'],
    stats['mean'],
    # yerr=stats['std'],                # or stats['std'] / np.sqrt(stats['count']) for SEM
    yerr = stats['mean'] / np.sqrt(stats['count']),
    fmt='o-',
    capsize=4,
    # label='Sample mean ± std'
    label='Sample mean ± mean/$\sqrt{N}$'
)
ax[0].axhline(4.29, label="theoretical freeflow mean", linestyle="--", color='gray')
ax[0].set_title('Mean laps_per_min vs num_robots')
ax[0].set_xlabel('num_robots')
ax[0].set_ylabel('mean(laps_per_min)')
ax[0].set_ylim([0, None])

ax[0].scatter(df_last["num_robots"], df_last["laps_per_min"], alpha = 0.1)

ax[0].legend()

# 2) Variance
sns.lineplot(data=stats, x='num_robots', y='var', marker='o', ax=ax[1])
ax[1].axhline(0.68, label="theoretical freeflow variance", linestyle="--", color='gray')
ax[1].set_title('Variance of laps_per_min vs num_robots')
ax[1].set_xlabel('num_robots')
ax[1].set_ylabel('Var(laps_per_min)')
ax[1].set_ylim([0, None])
ax[1].legend()

# 3) Coefficient of variation
sns.lineplot(data=stats, x='num_robots', y='cv', marker='o', ax=ax[2])
ax[2].set_title('Coefficient of variation vs num_robots')
ax[2].set_xlabel('num_robots')
ax[2].set_ylabel('CV = std / mean')
ax[2].set_ylim([0, None])

plt.tight_layout()
plt.show()

## Compare experiment and simulation

In [ ]:
sim_df = pd.read_csv(csv_path, low_memory=False)
sim_df["_time"] = sim_df["time"]
sim_df["laps_float"] = sim_df["lap_count_ccw"]
sim_df["target_speed_rad"] = sim_df["angular_speed"]
sim_df["max_speedup_acceleration"] = pd.to_numeric(sim_df["max_speedup_acceleration"], errors="coerce")
sim_df["reaction_delay"] = pd.to_numeric(sim_df["reaction_delay"], errors="coerce")
sim_df["accel_label"] = sim_df["max_speedup_acceleration"].map(lambda x: f"{x:.2f}" if pd.notnull(x) else "unlimited")
sim_df["reaction_delay_label"] = sim_df["reaction_delay"].map(lambda x: f"{x:.2f}")
sim_df["condition_label"] = (
    "accel=" + sim_df["accel_label"] + ", delay=" + sim_df["reaction_delay_label"]
)

sim_df_last = (
    sim_df
    .sort_values(["reaction_delay_label", "accel_label", "num_robots", "trial", "robot_id", "_time"])
    .groupby(["reaction_delay_label", "accel_label", "num_robots", "trial", "robot_id"], as_index=False)
    .tail(1)
)

sim_df_last["laps_per_min"] = sim_df_last["laps_float"] / EXPERIMENT_DURATION_MIN
sim_df_last["target_laps_per_min"] = sim_df_last["target_speed_rad"] / (2 * np.pi) * 60.0

sim_stats = (
    sim_df_last
    .dropna(subset=["accel_label", "reaction_delay_label", "num_robots", "laps_per_min"])
    .groupby(["reaction_delay_label", "accel_label", "num_robots"])
    .agg(
        mean=("laps_per_min", "mean"),
        var=("laps_per_min", "var"),
        std=("laps_per_min", "std"),
        count=("laps_per_min", "count"),
        distinct_runs=("trial", "nunique"),
    )
    .reset_index()
    .sort_values(["reaction_delay_label", "accel_label", "num_robots"])
)
sim_stats["cv"] = sim_stats["std"] / sim_stats["mean"]
sim_stats["condition_label"] = (
    "accel=" + sim_stats["accel_label"] + ", delay=" + sim_stats["reaction_delay_label"]
)

exp_df = pd.read_csv(
    "/Users/lucyliu/Documents/phd/traffic/robots/2026_passing_experiments/analysis/data_from_influx/influx_plus_recovery_notfiltered_20260511_1s.csv",
    low_memory=False,
)
exp_df = exp_df[~exp_df["run_uuid"].isin(disconnection_uuids + jam_uuids)].copy()
exp_df["_time"] = pd.to_datetime(exp_df["_time"])
exp_df["num_robots"] = exp_df["num_robots"].astype(int)
exp_df["r_current"] = np.sqrt(exp_df["x"] ** 2 + exp_df["y"] ** 2)

exp_df_last = (
    exp_df
    .sort_values(["run_uuid", "robot_id", "_time"])
    .groupby(["run_uuid", "robot_id"], as_index=False)
    .tail(1)
)

exp_df_last["laps_per_min"] = exp_df_last["laps_float"] / 5.0
exp_df_last["target_laps_per_min"] = exp_df_last["target_speed_rad"] / (2 * np.pi) * 60.0

exp_stats = (
    exp_df_last
    .dropna(subset=["num_robots", "laps_per_min"])
    .groupby("num_robots")
    .agg(
        mean=("laps_per_min", "mean"),
        var=("laps_per_min", "var"),
        std=("laps_per_min", "std"),
        count=("laps_per_min", "count"),
        distinct_runs=("run_uuid", "nunique"),
    )
    .reset_index()
    .sort_values("num_robots")
)
exp_stats["cv"] = exp_stats["std"] / exp_stats["mean"]

print(f"sim_df_last rows: {len(sim_df_last):,}")
print(f"exp_df_last rows: {len(exp_df_last):,}")
sim_stats.head(), exp_stats.head()

In [ ]:
condition_order = [
    f"accel={accel}, delay={delay}"
    for delay in REACTION_DELAY_ORDER
    for accel in ACCELERATION_ORDER
]
palette = dict(zip(condition_order, sns.color_palette("tab10", n_colors=len(condition_order))))

fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharex=True)
fig.suptitle("Experiment vs Simulation: Laps Per Minute Summary Statistics", fontsize=14)

for condition_label in condition_order:
    condition_stats = sim_stats[sim_stats["condition_label"] == condition_label].sort_values("num_robots")
    color = palette[condition_label]

    ax[0].errorbar(
        condition_stats["num_robots"],
        condition_stats["mean"],
        yerr=condition_stats["mean"] / np.sqrt(condition_stats["count"]),
        fmt="o-",
        capsize=4,
        color=color,
        label=f"{condition_label} sim",
    )
    condition_points = sim_df_last[sim_df_last["condition_label"] == condition_label]
    ax[0].scatter(condition_points["num_robots"], condition_points["laps_per_min"], alpha=0.06, color=color)

    sns.lineplot(
        data=condition_stats,
        x="num_robots",
        y="var",
        marker="o",
        ax=ax[1],
        color=color,
        label=f"{condition_label} sim",
    )

    sns.lineplot(
        data=condition_stats,
        x="num_robots",
        y="cv",
        marker="o",
        ax=ax[2],
        color=color,
        label=f"{condition_label} sim",
    )

ax[0].errorbar(
    exp_stats["num_robots"],
    exp_stats["mean"],
    yerr=exp_stats["mean"] / np.sqrt(exp_stats["count"]),
    fmt="s--",
    capsize=4,
    color="black",
    label="experiment",
)
ax[0].scatter(exp_df_last["num_robots"], exp_df_last["laps_per_min"], alpha=0.12, color="black")

sns.lineplot(
    data=exp_stats,
    x="num_robots",
    y="var",
    marker="s",
    linestyle="--",
    ax=ax[1],
    color="black",
    label="experiment",
)

sns.lineplot(
    data=exp_stats,
    x="num_robots",
    y="cv",
    marker="s",
    linestyle="--",
    ax=ax[2],
    color="black",
    label="experiment",
)

ax[0].axhline(theoretical_mean, label="theoretical freeflow mean", linestyle="--", color="gray")
ax[0].set_title("Mean laps_per_min vs num_robots")
ax[0].set_xlabel("num_robots")
ax[0].set_ylabel("mean(laps_per_min)")
ax[0].set_ylim([0, None])
ax[0].legend(fontsize=7)

ax[1].axhline(theoretical_variance, label="theoretical freeflow variance", linestyle="--", color="gray")
ax[1].set_title("Variance of laps_per_min vs num_robots")
ax[1].set_xlabel("num_robots")
ax[1].set_ylabel("Var(laps_per_min)")
ax[1].set_ylim([0, None])
ax[1].legend(fontsize=7)

ax[2].set_title("Coefficient of variation vs num_robots")
ax[2].set_xlabel("num_robots")
ax[2].set_ylabel("CV = std / mean")
ax[2].set_ylim([0, None])
ax[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
comparison_stats = sim_stats.merge(
    exp_stats,
    on=["num_robots"],
    how="left",
    suffixes=("_sim", "_exp"),
)
comparison_stats["mean_diff"] = comparison_stats["mean_sim"] - comparison_stats["mean_exp"]
comparison_stats["var_diff"] = comparison_stats["var_sim"] - comparison_stats["var_exp"]
comparison_stats["cv_diff"] = comparison_stats["cv_sim"] - comparison_stats["cv_exp"]
comparison_stats[[
    "reaction_delay_label",
    "accel_label",
    "num_robots",
    "mean_sim",
    "mean_exp",
    "mean_diff",
    "var_sim",
    "var_exp",
    "var_diff",
    "cv_sim",
    "cv_exp",
    "cv_diff",
]]

In [ ]:
fig, axes = plt.subplots(len(REACTION_DELAY_ORDER), len(ACCELERATION_ORDER), figsize=(14, 10), sharex=True, sharey=True)
axes = np.atleast_2d(axes)

xs = np.linspace(min_laps_per_min, max_laps_per_min, 50)

for row_idx, reaction_delay_label in enumerate(REACTION_DELAY_ORDER):
    for col_idx, accel_label in enumerate(ACCELERATION_ORDER):
        ax = axes[row_idx, col_idx]
        sim_plot_df = sim_df_last[
            (sim_df_last["reaction_delay_label"] == reaction_delay_label)
            & (sim_df_last["accel_label"] == accel_label)
        ]

        sns.scatterplot(
            data=sim_plot_df,
            x="target_laps_per_min",
            y="laps_per_min",
            hue="num_robots",
            palette="tab20",
            alpha=0.35,
            marker="o",
            legend=False,
            ax=ax,
        )
        sns.scatterplot(
            data=exp_df_last,
            x="target_laps_per_min",
            y="laps_per_min",
            hue="num_robots",
            palette="tab20",
            alpha=0.9,
            marker="X",
            legend=(row_idx == 0 and col_idx == 0),
            ax=ax,
        )
        ax.plot(xs, xs, label="y = x", linestyle="--", color="gray")
        ax.set_title(f"accel = {accel_label}, delay = {reaction_delay_label} s")
        ax.set_xlabel("Target Laps Per Min")
        ax.set_ylabel("Actual Laps Per Min\n(For one robot over a full trial)")
        ax.set_ylim([0, None])

        if row_idx == 0 and col_idx == 0:
            handles, labels = ax.get_legend_handles_labels()
            source_handles = [
                plt.Line2D([0], [0], marker="o", color="black", linestyle="", alpha=0.35, label="simulation"),
                plt.Line2D([0], [0], marker="X", color="black", linestyle="", alpha=0.9, label="experiment"),
            ]
            ax.legend(handles + source_handles, labels + ["simulation", "experiment"], title="num_robots / source", ncol=2, fontsize=8)
        else:
            legend = ax.get_legend()
            if legend is not None:
                legend.remove()

fig.suptitle("Experiment vs Simulation: Target vs Actual Laps Per Min", fontsize=14)
plt.tight_layout()
plt.show()